# Lock v4, Amendment 1: within-anchor reliability and corrected power expectation

**Estimation pool only. Runs before the confirmation notebook exists.**

The v4 estimation run measured the reliability of C *between pairs* (Spearman over 55 pairs). The primary endpoint is a *within-anchor* rank correlation over 10 partners, so the reliability that governs it is within-anchor, which the between-pair figure overstates (it includes anchor-level variance). The v4 power table also used C's reliability as a stand-in for the outcome's reliability, and hardcoded two implied-requirement values that the run contradicts.

This notebook measures, on the estimation pool only:
1. within-anchor split-half reliability of C and of the outcome (PDR misses avoided, union rule), at confirmation size;
2. a split-half **pseudo-confirmation preview**: C on one half against the outcome on the other half, within anchor, plus the best-marginal comparator and the partial correlation;
3. patients implied for within-anchor reliability of 0.80 and 0.90;
4. which model each seed ranks best on marginal cost (to interpret the 0.000 seed agreement for best_partner_marginal).

It writes `04_results/locked_config_v4_amendment1.json`, chained to the sha256 of `locked_config_v4.json`, which it does not modify. **One locked element changes, and only if `OUTCOME_DEFINITION` is left at `'anchor_relative'`**: the outcome becomes PDR eyes missed by the anchor that the partner catches, instead of gain over the better of the two. Set that flag before running. Everything else (PDR grade, predictor C, rules, uncertainty method, success criterion) stands; the power expectation is corrected.

In [1]:
# Cell 1: environment and locked parameters (identical to the v4 estimation notebook)
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, itertools, datetime
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata

BASE    = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS  = f'{BASE}/01_splits'
PREDS   = f'{BASE}/03_predictions'
RESULTS = f'{BASE}/04_results'
AUDIT   = f'{BASE}/99_audit'
SPLIT_FILE = f'{SPLITS}/eyepacs_split_kaggletest_estconf_v2.csv'
LOCK_FILE  = f'{RESULTS}/locked_config_v4.json'

# Guard: this amendment is only legitimate before confirmation is opened.
assert not os.path.exists(f'{RESULTS}/confirmation_results_v4.json'), 'Confirmation already opened; an amendment now would be post hoc.'
assert os.path.exists(LOCK_FILE), 'locked_config_v4.json missing.'

BURDEN = 0.25
REFERRAL_UNIT = 'eye'
W = {4: 500, 3: 200, 2: 15}
W_FP = 1
N_SPLITHALF = 50
RNG_SEED = 20260923

# Outcome definition for the primary. Set BEFORE running; it is recorded in the amendment.
#   'anchor_relative': PDR eyes missed by the anchor that the partner catches (m_A - m_both). Matches the
#                      selection question (anchor A is deployed; which second reader?) and the single-reader comparator.
#   'pair_min'       : v4 as locked, min(m_A, m_B) - m_both, the gain over the better of the two. Penalises a partner
#                      stronger than the anchor by construction, because that partner becomes the baseline.
OUTCOME_DEFINITION = 'anchor_relative'
assert OUTCOME_DEFINITION in ('anchor_relative', 'pair_min')

ROSTER = ['densenet121_binary','densenet121_5class',
          'efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class',
          'vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
SEED_TAGS = ['', '_seed1', '_seed2']
PAIRS = list(itertools.combinations(ROSTER, 2))
PAIRSET = set(PAIRS)
def pkey(a, c): return (a, c) if (a, c) in PAIRSET else (c, a)

with open(LOCK_FILE) as f: LOCK = json.load(f)
assert LOCK['definitions']['referral_unit'] == REFERRAL_UNIT and LOCK['definitions']['burden'] == BURDEN
print('Lock v4 loaded, locked at', LOCK['locked_at_utc'])

Mounted at /content/drive
Lock v4 loaded, locked at 2026-09-23T12:21:35.047720+00:00


In [2]:
# Cell 2: estimation pool (same filter and guardrails as v4)
def pick_col(df, candidates, what):
    for c in candidates:
        if c in df.columns: return c
    raise AssertionError(f'Could not find the {what} column. Columns present: {list(df.columns)}')

split = pd.read_csv(SPLIT_FILE)
ID_COL    = pick_col(split, ['image_id','image','id','filename'], 'image id')
PAT_COL   = pick_col(split, ['patient_id','patient','pid','subject_id'], 'patient id')
GRADE_COL = pick_col(split, ['level','grade','true_grade','diagnosis'], 'grade')
POOL_COL  = pick_col(split, ['pool','split','set','assignment','estconf'], 'pool assignment')

pool_lc  = split[POOL_COL].astype(str).str.lower()
vals     = set(pool_lc.unique())
EST_KEY  = next(v for v in vals if v.startswith('est'))
CONF_KEY = next(v for v in vals if v.startswith('conf'))
est_split = split[pool_lc == EST_KEY].copy()
CONF_PATIENTS = set(split.loc[pool_lc == CONF_KEY, PAT_COL].astype(str))
CONF_IDS      = set(split.loc[pool_lc == CONF_KEY, ID_COL].astype(str))
assert not (set(est_split[PAT_COL].astype(str)) & CONF_PATIENTS)

est_split = est_split.sort_values(ID_COL).reset_index(drop=True)
EST_IDS   = est_split[ID_COL].astype(str).to_numpy()
pat       = est_split[PAT_COL].astype(str).to_numpy()
grade     = est_split[GRADE_COL].to_numpy().astype(int)
referable = grade >= 2
n_eyes    = len(grade)
pat_codes, pat_index = np.unique(pat, return_inverse=True)
n_pat = len(pat_codes)
WMISS = np.zeros(n_eyes)
for g, w in W.items(): WMISS[grade == g] = w
print(f'Estimation pool: {n_eyes:,} eyes, {n_pat:,} patients. Confirmation not loaded.')

Estimation pool: 35,718 eyes, 17,859 patients. Confirmation not loaded.


In [3]:
# Cell 3: predictions, with hashes checked against the lock
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''): h.update(blk)
    return h.hexdigest()

P = {}
for a in ROSTER:
    for t in SEED_TAGS:
        fn = f'eyepacs_predictions_kaggletest_{a}{t}_v2.csv'
        path = f'{PREDS}/{fn}'
        assert sha256_file(path) == LOCK['file_sha256'][fn], f'{fn} changed since lock v4.'
        df = pd.read_csv(path)
        idc  = pick_col(df, ['image_id','image','id','filename'], 'image id')
        prob = pick_col(df, ['predicted_probability','prob_referable','probability','prob','p_referable'], 'probability')
        df[idc] = df[idc].astype(str)
        df = df[df[idc].isin(set(EST_IDS))]
        assert not (set(df[idc]) & CONF_IDS)
        df = df.drop_duplicates(subset=idc).set_index(idc).reindex(EST_IDS)
        assert df[prob].notna().all()
        P[f'{a}{t}'] = df[prob].to_numpy(dtype=np.float64)
assert sha256_file(SPLIT_FILE) == LOCK['file_sha256'][os.path.basename(SPLIT_FILE)]
LOCK_SHA = sha256_file(LOCK_FILE)
print(f'Loaded {len(P)} models; all 34 hashes match lock v4. locked_config_v4.json sha256 {LOCK_SHA[:16]}...')

Loaded 33 models; all 34 hashes match lock v4. locked_config_v4.json sha256 9b78257ece485d02...


In [4]:
# Cell 4: definitions (copied from the v4 estimation notebook, eye-level referral)
def refer_mask(scores, burden=BURDEN, sub=None):
    idx = np.arange(n_eyes) if sub is None else np.asarray(sub)
    s = scores[idx]
    k = int(round(burden * len(idx)))
    order = np.argsort(-s, kind='stable')
    m = np.zeros(len(idx), bool); m[order[:k]] = True
    return m

def m_miss(ref, w, rf):
    return float(w[rf & ~ref].sum())

def pair_quantities(a, b, sub=None):
    idx = np.arange(n_eyes) if sub is None else np.asarray(sub)
    w, rf, gr = WMISS[idx], referable[idx], grade[idx]
    C, pdr, ma_, mb_, mbo_ = [], [], [], [], []
    for ta in SEED_TAGS:
        ra = refer_mask(P[f'{a}{ta}'], sub=idx)
        for tb in SEED_TAGS:
            rb = refer_mask(P[f'{b}{tb}'], sub=idx)
            mA, mB = m_miss(ra, w, rf), m_miss(rb, w, rf)
            mboth = float(w[rf & ~ra & ~rb].sum()); mmin = min(mA, mB)
            C.append(1.0 - mboth / mmin if mmin > 0 else np.nan)
            sel = gr == 4
            xa, xb, xboth = int((sel & ~ra).sum()), int((sel & ~rb).sum()), int((sel & ~ra & ~rb).sum())
            pdr.append(min(xa, xb) - xboth); ma_.append(xa); mb_.append(xb); mbo_.append(xboth)
    return {'C': float(np.nanmean(C)), 'pdr_avoided': float(np.mean(pdr)),
            'pdr_miss_a': float(np.mean(ma_)), 'pdr_miss_b': float(np.mean(mb_)), 'pdr_both': float(np.mean(mbo_))}

def outcome(q, anchor, c, definition=None):
    definition = definition or OUTCOME_DEFINITION
    k = pkey(anchor, c); e = q[k]
    if definition == 'pair_min': return e['pdr_avoided']
    miss_anchor = e['pdr_miss_a'] if k[0] == anchor else e['pdr_miss_b']
    return miss_anchor - e['pdr_both']

def marginal_cost(a, sub=None):
    idx = np.arange(n_eyes) if sub is None else np.asarray(sub)
    w, rf = WMISS[idx], referable[idx]
    return float(np.mean([m_miss(refer_mask(P[f'{a}{t}'], sub=idx), w, rf) for t in SEED_TAGS]))

def within_anchor(pred, out, ctrl=None):
    '''Mean over anchors of the within-anchor Spearman (or rank partial, if ctrl given) across 10 partners.'''
    rs = []
    for anchor in ROSTER:
        partners = [c for c in ROSTER if c != anchor]
        x = rankdata([pred(anchor, c) for c in partners])
        y = rankdata([out(anchor, c) for c in partners])
        if ctrl is not None:
            z = rankdata([ctrl(anchor, c) for c in partners])
            Z = np.column_stack([np.ones_like(z), z])
            x = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
            y = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
        if np.std(x) > 0 and np.std(y) > 0:
            rs.append(float(np.corrcoef(x, y)[0, 1]))
    return float(np.mean(rs)) if rs else np.nan

def spearman_brown(r, k):
    return float(k * r / (1 + (k - 1) * r))

def patients_for(r_half, n_half, target):
    if not (0 < r_half < 1): return None
    k = target * (1 - r_half) / (r_half * (1 - target))
    return int(round(k * n_half))
print('Definitions loaded.')

Definitions loaded.


In [5]:
# Cell 5: split-half diagnostics, estimation pool only
def halves(seed):
    r = np.random.default_rng(seed)
    perm = r.permutation(n_pat)
    m1 = np.isin(pat_index, perm[: n_pat // 2])
    return np.where(m1)[0], np.where(~m1)[0]

acc = {k: [] for k in ('rel_C_between','rel_C_within','rel_out_within',
                       'preview_C','preview_marginal','preview_C_partial',
                       'alt_rel_out_within','alt_preview_C','alt_preview_marginal','alt_preview_C_partial')}
ALT = 'pair_min' if OUTCOME_DEFINITION == 'anchor_relative' else 'anchor_relative'
for rep in range(N_SPLITHALF):
    i1, i2 = halves(RNG_SEED + rep)   # same halves as the v4 reliability diagnostic
    q1 = {p: pair_quantities(*p, sub=i1) for p in PAIRS}
    q2 = {p: pair_quantities(*p, sub=i2) for p in PAIRS}
    m1 = {a: marginal_cost(a, sub=i1) for a in ROSTER}
    m2 = {a: marginal_cost(a, sub=i2) for a in ROSTER}

    acc['rel_C_between'].append(spearmanr([q1[p]['C'] for p in PAIRS], [q2[p]['C'] for p in PAIRS], nan_policy='omit').statistic)
    acc['rel_C_within'].append(within_anchor(lambda a, c: q1[pkey(a, c)]['C'], lambda a, c: q2[pkey(a, c)]['C']))
    # Pseudo-confirmation: predictor from one half, outcome from the other, both directions averaged.
    for pre, dfn in (('', OUTCOME_DEFINITION), ('alt_', ALT)):
        acc[pre + 'rel_out_within'].append(within_anchor(lambda a, c: outcome(q1, a, c, dfn), lambda a, c: outcome(q2, a, c, dfn)))
        pc, pm, pp = [], [], []
        for qa, qb, ma in ((q1, q2, m1), (q2, q1, m2)):
            pc.append(within_anchor(lambda a, c: qa[pkey(a, c)]['C'], lambda a, c: outcome(qb, a, c, dfn)))
            pm.append(within_anchor(lambda a, c: -ma[c], lambda a, c: outcome(qb, a, c, dfn)))
            pp.append(within_anchor(lambda a, c: qa[pkey(a, c)]['C'], lambda a, c: outcome(qb, a, c, dfn),
                                    ctrl=lambda a, c: -ma[c]))
        acc[pre + 'preview_C'].append(np.mean(pc)); acc[pre + 'preview_marginal'].append(np.mean(pm))
        acc[pre + 'preview_C_partial'].append(np.mean(pp))
    if rep % 10 == 9: print(f'  {rep + 1}/{N_SPLITHALF} repeats')

res = {k: float(np.nanmean(v)) for k, v in acc.items()}
sd  = {k: float(np.nanstd(v)) for k, v in acc.items()}
for k in res: print(f'{k:20s} {res[k]: .3f}  (sd over splits {sd[k]:.3f})')

  10/50 repeats
  20/50 repeats
  30/50 repeats
  40/50 repeats
  50/50 repeats
rel_C_between         0.589  (sd over splits 0.145)
rel_C_within          0.558  (sd over splits 0.117)
rel_out_within        0.556  (sd over splits 0.118)
preview_C             0.457  (sd over splits 0.078)
preview_marginal      0.242  (sd over splits 0.074)
preview_C_partial     0.519  (sd over splits 0.092)
alt_rel_out_within    0.594  (sd over splits 0.124)
alt_preview_C         0.317  (sd over splits 0.072)
alt_preview_marginal -0.494  (sd over splits 0.076)
alt_preview_C_partial  0.381  (sd over splits 0.083)


In [6]:
# Cell 6: corrected power expectation
n_half = n_pat / 2
K_CONF = len(CONF_PATIENTS) / n_half

r_C_half   = res['rel_C_within']
r_out_half = res['rel_out_within']
R_C_est    = spearman_brown(r_C_half, 2.0) if r_C_half > 0 else np.nan
R_out_conf = spearman_brown(r_out_half, K_CONF) if r_out_half > 0 else np.nan

TRUE_V3 = LOCK['power_expectation']['v3_disattenuated_underlying_estimate']
THRESH  = LOCK['power_expectation']['detection_threshold_eleven_anchors']
exp_obs_from_v3 = TRUE_V3 * float(np.sqrt(max(R_C_est, 0) * max(R_out_conf, 0)))

# The preview uses a half-pool predictor; scale it to the full-estimation predictor.
preview_scaled = res['preview_C'] * float(np.sqrt(max(R_C_est, 0) / r_C_half)) if r_C_half > 0 else np.nan

implied = {'within_anchor_C_rel_0.80_patients': patients_for(r_C_half, n_half, 0.80),
           'within_anchor_C_rel_0.90_patients': patients_for(r_C_half, n_half, 0.90),
           'between_pair_C_rel_0.80_patients': patients_for(res['rel_C_between'], n_half, 0.80),
           'between_pair_C_rel_0.90_patients': patients_for(res['rel_C_between'], n_half, 0.90)}

print(f'Within-anchor reliability of C: split-half {r_C_half:.3f}, full estimation {R_C_est:.3f}')
print(f'Within-anchor reliability of outcome at confirmation size: {R_out_conf:.3f}')
print(f'Expected observed primary if underlying = {TRUE_V3}: {exp_obs_from_v3:.3f}  (detection threshold {THRESH})')
print(f'Pseudo-confirmation preview (half predictor): {res["preview_C"]:.3f}; scaled to full predictor: {preview_scaled:.3f}')
print(f'Best-marginal comparator preview: {res["preview_marginal"]:.3f}; C partial on marginal: {res["preview_C_partial"]:.3f}')
print('Implied patients:', implied)

Within-anchor reliability of C: split-half 0.558, full estimation 0.716
Within-anchor reliability of outcome at confirmation size: 0.556
Expected observed primary if underlying = 0.33: 0.208  (detection threshold 0.28)
Pseudo-confirmation preview (half predictor): 0.457; scaled to full predictor: 0.518
Best-marginal comparator preview: 0.242; C partial on marginal: 0.519
Implied patients: {'within_anchor_C_rel_0.80_patients': 28309, 'within_anchor_C_rel_0.90_patients': 63694, 'between_pair_C_rel_0.80_patients': 24912, 'between_pair_C_rel_0.90_patients': 56052}


In [7]:
# Cell 7: which model each seed ranks best on marginal cost (interprets best_partner_marginal agreement = 0.000)
rank_tbl = {}
for t in SEED_TAGS:
    costs = {a: m_miss(refer_mask(P[f'{a}{t}']), WMISS, referable) for a in ROSTER}
    rank_tbl[t or '_seed0'] = sorted(costs.items(), key=lambda kv: kv[1])[:3]
for t, top in rank_tbl.items():
    print(t, '  '.join(f'{a} {c:,.0f}' for a, c in top))
top_by_seed = {t: v[0][0] for t, v in rank_tbl.items()}
gap_top2 = {t: (v[1][1] - v[0][1]) / v[0][1] for t, v in rank_tbl.items()}
print('Top model by seed:', top_by_seed)
print('Relative cost gap between the top two, by seed:', {k: round(x, 4) for k, x in gap_top2.items()})

_seed0 retfound_binary 54,595  retfound_5class 54,925  retfound_adversarial 56,960
_seed1 densenet121_binary 52,205  retfound_5class 53,660  retfound_binary 54,145
_seed2 densenet121_5class 51,810  retfound_5class 51,975  retfound_binary 52,410
Top model by seed: {'_seed0': 'retfound_binary', '_seed1': 'densenet121_binary', '_seed2': 'densenet121_5class'}
Relative cost gap between the top two, by seed: {'_seed0': 0.006, '_seed1': 0.0279, '_seed2': 0.0032}


In [8]:
# Cell 8: write the amendment (does not touch locked_config_v4.json)
NOW = datetime.datetime.now(datetime.timezone.utc).isoformat()
amend = {
  'amendment': 'v4.1',
  'amends': 'locked_config_v4.json',
  'amends_sha256': LOCK_SHA,
  'written_at_utc': NOW,
  'confirmation_status': 'CLOSED; not read by this notebook',
  'outcome_definition': OUTCOME_DEFINITION,
  'outcome_definition_changed_from_v4': OUTCOME_DEFINITION != 'pair_min',
  'outcome_definition_rationale': ('anchor_relative counts PDR eyes missed by the deployed anchor that the partner catches. '
      'The v4 pair_min outcome measures gain over the better of the two readers, which penalises a partner stronger than '
      'the anchor by construction and so biases the decision-relevant secondary against best_partner_marginal.'),
  'unchanged': ['PDR as the outcome grade (diagnostic 3)', 'predictor C and competing rules', 'uncertainty method',
                'success criterion', 'reliability gate verdict (failed)'],
  'reason': ('The v4 power expectation used between-pair reliability of C, whereas the primary is a within-anchor '
             'rank correlation; used C reliability in place of outcome reliability; and hardcoded implied requirements '
             '(25,000 patients for 0.90; patient axis satisfied) that the v4 run itself contradicts.'),
  'within_anchor_reliability': {'C_split_half': round(r_C_half, 4), 'C_full_estimation': round(R_C_est, 4),
                                'outcome_confirmation_size': round(R_out_conf, 4),
                                'C_between_pair_split_half_reproduced': round(res['rel_C_between'], 4)},
  'corrected_power_expectation': {
      'underlying_assumed_from_v3': TRUE_V3,
      'expected_observed_primary': round(exp_obs_from_v3, 3),
      'detection_threshold_eleven_anchors': THRESH,
      'expected_below_threshold': bool(exp_obs_from_v3 < THRESH),
      'interpretation': ('Expected observed effect is below the detection threshold. A null is a power statement and '
                         'does not rule out a relationship of the size v3 implies.' if exp_obs_from_v3 < THRESH else
                         'Expected observed effect reaches the detection threshold.')},
  'pseudo_confirmation_preview_estimation_only': {
      'note': 'Split-half within the estimation pool: predictor on one half, outcome on the other. Recorded before confirmation as the empirical expectation.',
      'C_vs_outcome_half_predictor': round(res['preview_C'], 4),
      'C_vs_outcome_scaled_to_full_predictor': round(preview_scaled, 4),
      'best_marginal_vs_outcome': round(res['preview_marginal'], 4),
      'C_partial_on_partner_marginal': round(res['preview_C_partial'], 4),
      'alternative_outcome': ALT,
      'alt_outcome_reliability_split_half': round(res['alt_rel_out_within'], 4),
      'alt_C_vs_outcome_half_predictor': round(res['alt_preview_C'], 4),
      'alt_best_marginal_vs_outcome': round(res['alt_preview_marginal'], 4),
      'alt_C_partial_on_partner_marginal': round(res['alt_preview_C_partial'], 4),
      'sd_over_splits': {k: round(v, 4) for k, v in sd.items()}},
  'implied_requirements_corrected': {**implied,
      'estimation_pool_patients': n_pat,
      'patient_axis_satisfied_for_0.80': bool(implied['within_anchor_C_rel_0.80_patients'] is not None and n_pat >= implied['within_anchor_C_rel_0.80_patients']),
      'architecture_axis_satisfied': False,
      'supersedes': 'power_expectation.implied_requirements in locked_config_v4.json'},
  'best_partner_marginal_seed_agreement_note': {
      'top_model_by_seed': top_by_seed, 'relative_gap_top_two_by_seed': {k: round(x, 4) for k, x in gap_top2.items()},
      'distinct_top_models': len(set(top_by_seed.values())),
      'interpretation': (('best_partner_marginal picks the same partner for nearly every anchor, so its seed agreement is '
                          'all-or-nothing and the 0.10 chance line does not apply. Three different top models across three '
                          'seeds accounts for the v4 figure of 0.000.') if len(set(top_by_seed.values())) == 3 else
                         ('INCONSISTENT: at least two seeds share a top model, which should give best_partner_marginal '
                          'agreement well above 0.000. The v4 diagnostic 4 figure for this rule is suspect and must be '
                          'rechecked before confirmation.'))}}

with open(f'{RESULTS}/locked_config_v4_amendment1.json', 'w') as f: json.dump(amend, f, indent=2)

entry = f'''

## Lock v4 amendment 1 ({NOW[:10]})

Estimation pool only; confirmation not read. Chained to locked_config_v4.json (sha256 {LOCK_SHA[:16]}...).
Within-anchor C reliability {R_C_est:.3f} (full estimation), outcome reliability {R_out_conf:.3f} (confirmation size).
Corrected expected observed primary {exp_obs_from_v3:.3f} against threshold {THRESH}. Pseudo-confirmation preview
{res["preview_C"]:.3f} (half predictor), best-marginal {res["preview_marginal"]:.3f}, partial {res["preview_C_partial"]:.3f}.
Outcome definition: {OUTCOME_DEFINITION}. Rules, uncertainty method and success criterion unchanged. Produced by 99_audit/AMEND_lock_v4_within_anchor.ipynb.
'''
with open(f'{AUDIT}/RUN_LOG.md', 'a') as f: f.write(entry)
print(json.dumps(amend, indent=2))

{
  "amendment": "v4.1",
  "amends": "locked_config_v4.json",
  "amends_sha256": "9b78257ece485d0266b222a69e48f8eaa94f7b351911238d16195a5b087eb6cc",
  "written_at_utc": "2026-09-23T13:04:28.343103+00:00",
  "confirmation_status": "CLOSED; not read by this notebook",
  "outcome_definition": "anchor_relative",
  "outcome_definition_changed_from_v4": true,
  "outcome_definition_rationale": "anchor_relative counts PDR eyes missed by the deployed anchor that the partner catches. The v4 pair_min outcome measures gain over the better of the two readers, which penalises a partner stronger than the anchor by construction and so biases the decision-relevant secondary against best_partner_marginal.",
  "unchanged": [
    "PDR as the outcome grade (diagnostic 3)",
    "predictor C and competing rules",
    "uncertainty method",
    "success criterion",
    "reliability gate verdict (failed)"
  ],
  "reason": "The v4 power expectation used between-pair reliability of C, whereas the primary is a wit

## After this notebook
Send back `04_results/locked_config_v4_amendment1.json`. Move this executed notebook to `99_audit/AMEND_lock_v4_within_anchor.ipynb`. The confirmation notebook is built only after the amendment exists, and it asserts both lock files are present and unchanged.